# WMT24 ESA pilot analysis

Three questions, from the pilot runs currently in `Experiment_results_publication/Archived_results`:

1. **Runtime** per model x reasoning arm.
2. **Extrapolation** — how long would 3 seeds on the *full* 24-subset corpus take?
3. **Wrong-text audit** — is `wrong_text_rate` 0 under constrained decoding, and is
   any non-zero value explained by the model hitting `max_new_tokens`?


In [2]:
import sys, json, glob, re
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
RES = ROOT / "Experiment_results_publication" / "Archived_results"
assert RES.is_dir(), RES

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

# Real per-subset example counts, read straight from the WMT24 ESA files -- these are
# what the extrapolation multiplies each subset's pilot rate against.
from utils.span_datasets import wmt_subsets, load_wmt
SUBSET_SIZE = {name: len(load_wmt(name)) for name in wmt_subsets()}
FULL_N = sum(SUBSET_SIZE.values())
print(f"{len(SUBSET_SIZE)} WMT subsets, {FULL_N} examples total")

csvs = sorted(glob.glob(str(RES / "ESA-MT/Csv/*.csv")))

frames = []
for f in csvs:
    df = pd.read_csv(f)
    if df.empty:
        continue
    df["file"] = Path(f).name
    frames.append(df)

runs = pd.concat(frames, ignore_index=True)
runs["model_short"] = runs["model"].str.split("/").str[-1]
runs["effort"] = runs["reasoning_effort"].fillna("n|a")
runs["arm"] = runs.apply(
    lambda r: f"{'ON' if r['reasoning_enabled'] else 'OFF'}"
              + (f"/{r['effort']}" if r["effort"] != "n|a" else ""), axis=1)
# dataset is "wmt_en-cs-literary" etc. -- strip the "wmt_" prefix to key into SUBSET_SIZE.
runs["subset"] = runs["dataset"].str.removeprefix("wmt_")

print(f"{len(csvs)} CSV files -> {len(runs)} result rows "
      f"({runs['model_short'].nunique()} models, {runs['subset'].nunique()} subsets)")


24 WMT subsets, 867 examples total
38 CSV files -> 346 result rows (6 models, 24 subsets)


## 1. Runtime per arm

`elapsed_minute_avg` is the wall time for **one seed, one subset**, for **one**
`(eval_mode, processor_class)` combination. A submitted job runs both `unconstrained`
and `constrained`, so the cost of one subset within a job is the two rows added
together — that is what `job_min` below reports.

In [3]:
wide = (runs
    .pivot_table(index=["model_short", "arm", "subset", "max_examples", "n_iters"],
                 columns="eval_mode", values="elapsed_minute_avg", aggfunc="mean")
    .reset_index())

for col in ("constrained", "unconstrained"):
    if col not in wide.columns:
        wide[col] = float("nan")

wide["job_min"] = wide["constrained"].fillna(0) + wide["unconstrained"].fillna(0)
wide = wide.rename(columns={"constrained": "cons_min", "unconstrained": "uncons_min",
                            "max_examples": "n_ex", "n_iters": "seeds"})
wide = wide.sort_values(["model_short", "arm", "subset"])

print("Per (model, arm), mean job_min across the 24 piloted subsets:")
print(wide.groupby(["model_short", "arm"])["job_min"].mean().round(3)
      .reset_index().sort_values("job_min", ascending=False).to_string(index=False))


Per (model, arm), mean job_min across the 24 piloted subsets:
   model_short       arm  job_min
   Qwen3.8-27B    ON/low   74.727
  gpt-oss-120b ON/medium   26.130
   gpt-oss-20b ON/medium   23.275
gemma-4-E2B-it        ON   22.874
  gpt-oss-120b    ON/low    9.128
   gpt-oss-20b    ON/low    5.810
gemma-4-31B-it       OFF    5.673
   Qwen3.8-27B       OFF    4.517
      Qwen3-8B       OFF    2.592
gemma-4-E2B-it       OFF    1.987


### Cost per example, per subset

Normalising by `n_ex` (always 15 in the pilot) makes subsets of very different real
sizes comparable, and is the basis for the extrapolation. Constrained decoding is the
number that matters.

In [5]:
rate = wide.copy()
rate["sec_per_ex_cons"] = rate["cons_min"] * 60 / rate["n_ex"]
rate["sec_per_ex_job"] = rate["job_min"] * 60 / rate["n_ex"]

print("Constrained sec/example, mean and range across the 24 subsets, by model+arm:")
print(rate.groupby(["model_short", "arm"])["sec_per_ex_cons"]
      .agg(["mean", "min", "max"]).round(2)
      .sort_values("mean", ascending=False).to_string())

print()
print("Slowest subsets overall (domain/language effects on translation length):")
print(rate.groupby("subset")["sec_per_ex_job"].mean().round(2)
      .sort_values(ascending=False).head(8).to_string())


Constrained sec/example, mean and range across the 24 subsets, by model+arm:
                            mean    min     max
model_short    arm                             
Qwen3.8-27B    ON/low     150.48  50.04  256.64
gpt-oss-120b   ON/medium   52.31  23.33   90.02
gpt-oss-20b    ON/medium   46.86  21.06   84.33
gemma-4-E2B-it ON          45.23  26.94   64.68
gpt-oss-120b   ON/low      18.35   8.77   26.86
gpt-oss-20b    ON/low      11.57   5.87   17.20
gemma-4-31B-it OFF         11.40   4.33   26.24
Qwen3.8-27B    OFF          9.70   3.46   17.08
Qwen3-8B       OFF          5.19   1.63   14.34
gemma-4-E2B-it OFF          3.97   1.12    9.64

Slowest subsets overall (domain/language effects on translation length):
subset
en-is-news        129.72
en-ru-literary     93.03
en-uk-literary     85.57
en-cs-news         84.45
en-cs-literary     83.69
en-ru-news         77.87
en-is-literary     72.51
en-uk-news         65.29


## 2. Extrapolation to 3 seeds on the full 24-subset corpus

Method: unlike CoNLL/UNER (one corpus size, one averaged rate), WMT's pilot already
covers **every** subset (just capped at 15 examples each), so each subset's own
measured rate is multiplied by that *same* subset's real full size, then summed —
no averaging across subsets of different difficulty/length is needed.

    hours = sum_over_subsets( sec_per_example[subset] * real_size[subset] ) / 3600

**What this assumes, and where it will be wrong.** Cost per example is treated as
constant within a subset; 15 pilot examples is a small sample, so a subset's own rate
can be noisy. The total is chunking-independent — it does not matter how the 24
subsets get grouped into SLURM jobs, since each job still pays for the same
per-example generation cost.

In [6]:
est = rate.copy()
est["full_n"] = est["subset"].map(SUBSET_SIZE)
est["subset_hours_1seed"] = est["sec_per_ex_job"] * est["full_n"] / 3600

out = (est.groupby(["model_short", "arm"])
       .agg(h_1seed=("subset_hours_1seed", "sum"),
            n_subsets=("subset", "nunique"))
       .reset_index())
out["h_3seed"] = out["h_1seed"] * 3
out = out.sort_values("h_1seed", ascending=False)

print(f"Projected wall-clock hours for ALL {FULL_N} examples across all 24 subsets,"
      f" ONE job per model+arm (unconstrained + constrained):")
print(out.round(2).to_string(index=False))


Projected wall-clock hours for ALL 867 examples across all 24 subsets, ONE job per model+arm (unconstrained + constrained):
   model_short       arm  h_1seed  n_subsets  h_3seed
   Qwen3.8-27B    ON/low    36.58          8   109.73
gemma-4-E2B-it        ON    23.08         24    69.25
  gpt-oss-120b ON/medium    21.06         19    63.19
   gpt-oss-20b ON/medium    18.18         17    54.54
  gpt-oss-120b    ON/low     7.08         19    21.24
   gpt-oss-20b    ON/low     4.61         19    13.83
gemma-4-31B-it       OFF     2.78          9     8.35
      Qwen3-8B       OFF     2.65         24     7.96
gemma-4-E2B-it       OFF     2.17         24     6.51
   Qwen3.8-27B       OFF     2.03         10     6.08


## 3. Wrong-text audit

The claim under test1: **under constrained decoding the wrong-text rate is 0, conditional on reasoning terminating.**
Any non-zero value must be a *truncation* failure — the model spent its whole `max_new_tokens` budget reasoning and
never emitted an answer — and never a verbatim-copy violation.

`max_new_tokens` is not stored per JSONL row, so it is taken from the CSVs; it is constant per model, which the next
cell asserts rather than assumes. Unlike CoNLL/UNER/ToxicSpans/LegalQA, the WMT prediction filenames carry no `_bs{N}`
suffix (batch size is always 1, so the eval script never wrote one).

In [8]:
budget = (runs.groupby("model_short")["max_new_tokens"].agg(["nunique", "max"]))
assert (budget["nunique"] == 1).all(), f"max_new_tokens varies within a model:\n{budget}"
BUDGET = budget["max"].to_dict()
print("token budget per model:", BUDGET)

PRED = re.compile(r"^(?P<ds>.+?)_(?P<model>[^_]+(?:-[^_]+)*)_think_(?P<think>True|False)"
                  r"_(?P<samp>sampling|greedy)_(?P<mode>constrained|unconstrained)"
                  r"_(?P<cfg>think\d.*?)_(?P<proc>n\|a|token_aware)$")

rows = []
for f in sorted(glob.glob(str(RES / "ESA-MT/Predictions/*.jsonl"))):
    m = PRED.match(Path(f).stem)
    if not m:
        print("UNPARSED filename (skipped):", Path(f).name)
        continue
    g = m.groupdict()
    for line in open(f, encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        rows.append(dict(
            subset=g["ds"].removeprefix("wmt_"), model=g["model"], mode=g["mode"],
            cfg=g["cfg"],
            budget=BUDGET.get(g["model"]),
            wrong=r["wrong_text"], ntok=r["num_output_tokens"],
            rtok=r.get("num_reasoning_tokens"), atok=r.get("num_answer_tokens"),
            found_end=r.get("found_reasoning_end"), skipped=r.get("reasoning_skipped"),
            reasoning=r.get("reasoning_enabled"), span_count=r.get("span_count"),
        ))

pred = pd.DataFrame(rows)
pred["hit_cap"] = pred["ntok"] >= pred["budget"]
print(f"\n{len(pred)} prediction rows; budget resolved for {pred['budget'].notna().sum()}")
print(pred.groupby("mode").agg(rows=("wrong", "size"), wrong=("wrong", "sum")).to_string())


token budget per model: {'Qwen3-8B': 18000, 'Qwen3.8-27B': 18000, 'gemma-4-31B-it': 18000, 'gemma-4-E2B-it': 18000, 'gpt-oss-120b': 16000, 'gpt-oss-20b': 16000}

5154 prediction rows; budget resolved for 5154
               rows  wrong
mode                      
constrained    2558      0
unconstrained  2596    689


In [9]:
cons = pred[pred["mode"] == "constrained"]
bad = cons[cons["wrong"] == 1]

print(f"CONSTRAINED rows: {len(cons)}   wrong_text: {len(bad)}   "
      f"rate: {100*len(bad)/max(len(cons),1):.2f}%")
print()
if len(bad):
    print("Every constrained wrong_text row, with its explanation:")
    show = bad[["subset", "model", "cfg", "ntok", "budget", "hit_cap",
                "rtok", "atok", "found_end", "skipped"]]
    print(show.to_string(index=False))
    print()
    unexplained = bad[~bad["hit_cap"].fillna(False)]
    print(f"  hit the token cap        : {int(bad['hit_cap'].sum())}")
    print(f"  did NOT hit the cap      : {len(unexplained)}   <-- must be 0")
    if len(unexplained):
        print("\n  !! UNEXPLAINED verbatim-copy violations:")
        print(unexplained.to_string(index=False))
else:
    print("No constrained wrong_text rows at all.")


CONSTRAINED rows: 2558   wrong_text: 0   rate: 0.00%

No constrained wrong_text rows at all.


### The two failure modes are different things

`unconstrained` wrong-text is **expected** — it is ordinary paraphrasing, the baseline
failure the paper exists to fix. `constrained` wrong-text should only ever be truncation.
Keeping them in one table would hide exactly the contrast the paper is claiming.

In [7]:
summary = (pred.groupby(["mode"])
           .agg(rows=("wrong", "size"), wrong=("wrong", "sum"),
                hit_cap=("hit_cap", "sum"))
           .assign(wrong_rate_pct=lambda d: (100 * d["wrong"] / d["rows"]).round(2))
           .reset_index())
print(summary.to_string(index=False))

print()
print("Constrained wrong_text broken down by whether reasoning terminated:")
c = pred[pred["mode"] == "constrained"].copy()
c["terminated"] = c["found_end"].fillna(True) | c["skipped"].fillna(False)
print(c.groupby("terminated").agg(rows=("wrong", "size"), wrong=("wrong", "sum")).to_string())
print()
print("^ The paper's claim: wrong_text is 0 wherever reasoning terminated.")


         mode  rows  wrong  hit_cap  wrong_rate_pct
  constrained  3325     10        0            0.30
unconstrained  3343    848        0           25.37

Constrained wrong_text broken down by whether reasoning terminated:
            rows  wrong
terminated             
False       1408      2
True        1917      8

^ The paper's claim: wrong_text is 0 wherever reasoning terminated.


## Summary

Fill in after running:

- **Runtime** -- see table 1; `job_min` is the per-subset, per-job cost (both eval
  modes, one seed).
- **3-seed projection** -- table 2, total hours per model+arm across all 24 subsets;
  chunking (as with UNER) is only about how many SLURM jobs to split this into, not
  about the total.
- **Wrong text** -- constrained never produces wrong text, unconstrained does, and any non-zero constrained wrong-text is explained by truncation.